1. Install & Imports

In [ ]:
!pip install transformers torch scikit-learn

import pandas as pd
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import BertTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')


2. Load Dataset





In [ ]:
df = pd.read_csv("twitter_training.csv", header=None)

df.columns = ['id', 'topic', 'sentiment', 'text']

df = df[['text', 'sentiment']]
df = df.dropna()

df.head()

3. Preprocessing (OPTIMIZED)

In [ ]:
# Convert labels
df['sentiment'] = df['sentiment'].map({
    'Negative': 0,
    'Neutral': 1,
    'Positive': 2
})

df = df.dropna()
df['sentiment'] = df['sentiment'].astype(int)

# Lowercase
df['text'] = df['text'].str.lower()

df = df.sample(800)

texts = df['text'].tolist()
labels = df['sentiment'].tolist()

4. Train-Test Split

In [ ]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.3, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)

5. Tokenization

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding='max_length', max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding='max_length', max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding='max_length', max_length=128)

6. Dataset Class

In [ ]:
class TwitterDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

7. DataLoader

In [ ]:
train_dataset = TwitterDataset(train_encodings, train_labels)
val_dataset = TwitterDataset(val_encodings, val_labels)
test_dataset = TwitterDataset(test_encodings, test_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

8. Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

9. Optimizer

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

10. Training Loop

In [ ]:
epochs = 1

for epoch in range(epochs):
    print(f"\nStarting Epoch {epoch+1}")

    model.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 20 == 0:
            print(f"Step {i}, Loss: {loss.item()}")

    print(f"Epoch {epoch+1} Loss: {total_loss}")

11. Evaluation

In [ ]:
def evaluate(model, loader):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    print("Classification Report:\n")
    print(classification_report(true_labels, predictions))

    print("\nConfusion Matrix:\n")
    print(confusion_matrix(true_labels, predictions))

12. Run Evaluation

In [ ]:
evaluate(model, test_loader)

13. Experiment

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

In [ ]:
df = df.sample(800)
max_length = 128
batch_size = 8
epochs = 1